# IVP Group 19: Improved Multi-View Action Recognition Architecture
### ResNet-18 + 2-Layer Temporal Transformer with Learned Attention Pooling & Cosine Consistency Loss

- **Benchmark Dataset**: NW-UCLA Multiview Action Dataset (10 Action Classes, 3 Views, 10 Subjects)
- **Target Benchmark Protocol**: Cross-View Protocol $\{1, 2\} \rightarrow 3$ (Unseen Camera View 3 Holdout)
- **Historical Baseline**: 36.60% (153/418) [Inception-v3 + GRU]
- **Measured Augmentation**: 39.47% (165/418) [Inception-v3 + TSN-GRU + Clip-Consistent Augmentation]
- **Hardware**: NVIDIA Tesla T4 GPU (CUDA 12.x / PyTorch 2.x)


In [ ]:
# 1. Environment Verification
import os, sys, re, random, time, json, csv, math
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. Dataset Constants & Action Definitions
ACTIONS = [1, 2, 3, 4, 5, 6, 8, 9, 11, 12]
ACTION_NAMES = [
    'Pick up with one hand', 'Pick up with two hands', 'Drop trash',
    'Walk around', 'Sit down', 'Stand up', 'Put on a jacket',
    'Take off a jacket', 'Throw', 'Carry'
]
ACTION_SHORT = ['Pick-1H', 'Pick-2H', 'Drop', 'Walk', 'Sit', 'Stand', 'Don', 'Doff', 'Throw', 'Carry']
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)


In [ ]:
# 3. Video Inventory and Dataset Preparation
def inventory(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Dataset directory not found: {root}')
    roots = [root] if root.name == 'multiview_action_videos' else list(root.rglob('multiview_action_videos'))
    if len(roots) != 1:
        raise ValueError(f'Expected one multiview_action_videos directory, found {roots}')
    pattern = re.compile(r'v(\d+)_s(\d+)_e(\d+)\.avi$', re.I)
    groups = defaultdict(dict)
    for p in sorted(roots[0].rglob('*.avi')):
        match = pattern.fullmatch(p.name)
        if not match or not re.fullmatch(r'a\d+', p.parent.name, re.I):
            continue
        a = int(p.parent.name[1:])
        v, s, e = map(int, match.groups())
        if a not in ACTIONS: continue
        groups[(a, s, e)][v] = str(p.resolve())
    rows = [
        {'action_id': a, 'label': ACTIONS.index(a), 'subject': s, 'event': e, 'paths': [views[i] for i in (1, 2, 3)]}
        for (a, s, e), views in sorted(groups.items()) if all(i in views for i in (1, 2, 3))
    ]
    print(f'Discovered {len(rows)} complete three-view groups across {len(ACTIONS)} actions.')
    return rows

def segment_indices(n, count, train):
    indices = []
    for i in range(count):
        lo = min(n - 1, i * n // count)
        hi = min(n - 1, max(lo, (i + 1) * n // count - 1))
        indices.append(random.randint(lo, hi) if train else (lo + hi) // 2)
    return indices

def read_clip(path, frames=8, size=224, train=False):
    cap = cv2.VideoCapture(str(path))
    try:
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = segment_indices(count, frames, train)
        flip = train and random.random() < 0.5
        jitter = train and random.random() < 0.5
        contrast = random.uniform(0.9, 1.1) if jitter else 1.0
        brightness = random.uniform(-8, 8) if jitter else 0.0
        output = []
        for index in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, index)
            ok, image = cap.read()
            if not ok: raise RuntimeError(f'Cannot decode frame {index} in {path}')
            image = cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (size, size))
            if flip: image = np.ascontiguousarray(image[:, ::-1])
            if jitter: image = np.clip(image.astype(np.float32) * contrast + brightness, 0, 255).astype(np.uint8)
            output.append(torch.from_numpy(((image.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1).copy()))
        return torch.stack(output)
    finally: cap.release()

class VideoGroups(Dataset):
    def __init__(self, groups, split, frames=8, size=224):
        self.groups = groups; self.split = split; self.frames = frames; self.size = size
    def __len__(self): return len(self.groups)
    def __getitem__(self, index):
        group = self.groups[index]
        views = (0, 1) if self.split in ('train', 'validation') else (2,)
        clips = torch.stack([read_clip(group['paths'][v], self.frames, self.size, self.split == 'train') for v in views])
        return clips, torch.tensor(group['label'], dtype=torch.long)


In [ ]:
# 4. View-Invariant Spatiotemporal Model Architecture
class ViewInvariantActionNet(nn.Module):
    def __init__(self, num_classes=10, frames=8, hidden=256, pretrained=True, dropout=0.2):
        super().__init__()
        self.frames = frames
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        encoder.fc = nn.Identity()
        self.encoder = encoder
        self.project = nn.Sequential(nn.Linear(512, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout))
        self.position = nn.Parameter(torch.zeros(1, frames, hidden))
        nn.init.trunc_normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=4, dim_feedforward=hidden*2, dropout=dropout, batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.attention = nn.Linear(hidden, 1)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def forward(self, clips):
        batch, steps, channels, height, width = clips.shape
        features = self.encoder(clips.reshape(batch * steps, channels, height, width))
        tokens = self.project(features).reshape(batch, steps, -1) + self.position
        tokens = self.temporal(tokens)
        weights = self.attention(tokens).softmax(dim=1)
        embedding = self.norm((tokens * weights).sum(dim=1))
        return self.classifier(embedding), embedding

def paired_loss(logits, embeddings, labels, consistency_weight=0.1):
    targets = labels[:, None].expand(-1, 2).reshape(-1)
    classification = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets, label_smoothing=0.05)
    alignment = (1 - F.cosine_similarity(embeddings[:, 0], embeddings[:, 1], dim=-1)).mean()
    return classification + consistency_weight * alignment, classification.detach(), alignment.detach()


In [ ]:
# 5. Evaluation and Metric Computation Functions
def metrics(targets, predictions, loss):
    cm = np.zeros((len(ACTIONS), len(ACTIONS)), dtype=np.int64)
    for target, pred in zip(targets, predictions): cm[target, pred] += 1
    tp = cm.diagonal().astype(float); support = cm.sum(axis=1); predicted = cm.sum(axis=0)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)
    return {
        'loss': float(loss),
        'accuracy': float(tp.sum() / cm.sum()),
        'macro_precision': float(precision.mean()),
        'macro_recall': float(recall.mean()),
        'macro_f1': float(f1.mean()),
        'support': int(cm.sum())
    }, cm

def evaluate(model, loader, device):
    model.eval(); targets = []; predictions = []; losses = 0
    with torch.inference_mode():
        for clips, labels in loader:
            b, v, t, c, h, w = clips.shape
            logits, _ = model(clips.to(device).reshape(b * v, t, c, h, w))
            probs = logits.reshape(b, v, -1).softmax(-1).mean(1)
            losses += nn.functional.nll_loss(probs.clamp_min(1e-9).log(), labels.to(device), reduction='sum').item()
            targets.extend(labels.tolist())
            predictions.extend(probs.argmax(-1).cpu().tolist())
    result, cm = metrics(targets, predictions, losses / len(targets))
    return result, cm, targets, predictions


In [ ]:
# 6. Execution Protocol: Development (20 Epochs) -> Refit (12 Epochs) -> Target View 3 Test
print('Ready for execution.')
print('Command to execute full experiment:')
print('!python train.py --data /kaggle/input/nwucla-rgb/multiview_action_videos --output runs/experiment_01 --epochs 20 --batch-size 2 --evaluate-test')
